## Third step

In `third_step.ipynb`, we will use the records filtered (or not) in `second_step.ipynb` and filter for the federal states of the country or countries relevant to our needs. Like the previous one, this step would be necessary to focus on a specific study region.

Following the same convention as the previous step, the new field will be named `stateprovince_upd`, since the original field is `stateprovince`.

If you do not wish to perform this filtering, continue using the `stateprovince` field instead of `stateprovince_upd` for subsequent filtering steps.

In [ ]:
from library import *

specieslink, db_config = configure()

In [ ]:
conn = mysql_conn.connect(**db_config)
cursor = conn.cursor()

requirement = "country_upd" # change for country if country_upd was not done in second_step
column = "stateprovince"
table = "biodiversity_records"

sql = f"""SELECT {column} FROM {table} WHERE {column} IS NOT NULL AND {requirement} IS NOT NULL GROUP BY {column}"""

cursor.execute(sql)
results = cursor.fetchall()

for result in results:
    print(result[0])

cursor.close()
conn.close()  

Once again, you will need to manually filter the values ​​you are looking for and populate them in the pipeline. In my case, I will group them by Brazilian states and consolidate them under a single name. I will use a mapping to update the values; if your results differ from mine, which is highly likely, update the mapping as follows:

STATE_MAP = {
"Name I want to keep": [
"Name variation 1",
"Name variation 2",
],
(...)
}

In [ ]:
STATE_MAP = {
    "Rio de Janeiro": [
        "Rio de Janeiro",
        "RJ",
    ],
    "Minas Gerais": [
        "Minas Gerais",
        "MG",
        "MINAS GERAIS, MG",
    ],
    "São Paulo": [
        "SP",
        "Sao Paulo",
        "São Paulo",
        "SÃ£o Paulo",
    ],
    "Santa Catarina": [
        "Santa Catarina",
        "SC",
    ],
    "Bahia": [
        "Bahia",
        "BA",
        "BAHIA, BA",
    ],
    "Goiás": [
        "Goias",
        "GO",
    ],
    "Mato Grosso": [
        "Mato Grosso",
        "MT",
    ],
    "Ceará": [
        "Ceará",
        "CearÃ¡",
        "CE",
    ],
    "Piaui": [
        "Piaui",
        "PI",
    ],
    "Maranhão": [
        "Maranhão",
        "MA",
       "MaranhÃ£o" ,
    ],
    "Amapá": [
        "Amapá",
    ],
    "Alagoas": [
        "Alagoas",
        "AL",
    ],
    "Pernambuco": [
        "Pernambuco",
        "PE",
        "Fernando de Noronha",
        "Pernanbuco",
        "Sairé",
    ],
    "Rio Grande do Norte": [
        "Rio Grande do Norte",
    ],
    "Sergipe": [
        "Sergipe",
    ],
    "Paraíba": [
        "Paraíba",
        "ParaÃ­ba",
        "PB",
    ],
    "Espírito Santo": [
        "Espírito Santo",
        "ES",
        "ESPÍRITO SANTO, ES",
    ],
    "Pará": [
        "Pará",
        "PA",
    ],
    "Acre": [
        "Acre",
        "Rio Branco",
        "AC",
    ],
    "Paraná": [
        "Parana",
        "Paraná",
        "ParanÃ¡",
        "PR",
    ],
    "Rio Grande do Sul": [
        "Rio Grande do Sul",
        "RS",
        "Alegrete",
    ],
    "Rondônia": [
        "Rondônia",
        "RO",
    ],
    "Distrito Federal": [
        "Distrito Federal",
        "DF",
        "Brasília",
    ],
    "Roraima": [
        "Roraima",
    ],
    "Amazonas": [
        "Amazonas",
    ],
    "Tocantins": [
        "Tocantins",
        "TO",
    ],
    "Mato Grosso do Sul": [
        "Mato Grosso do Sul",
        "MS",
        "Matogroso do Sul",
    ],
}


In [ ]:
def filtering(field_input, update_input, filters_input, table):
    if not field_input:
        print("please provide a field")
        return
    
    conn = mysql_conn.connect(**db_config)
    cursor = conn.cursor()

    try:
        sql = f"ALTER TABLE {table} ADD COLUMN {field_input} TEXT"

        cursor.execute(sql)
        conn.commit()

        print(f"field '{field_input}' created successfully on table '{table}'")

        cursor.close()
        conn.close()  
    except Exception as e:
        print(f"field '{field_input}' already exists or error creating: {e}")
        
    filters = {}
    if '=' not in filters_input:
        print("badly formatted filter: use field=value1,value2,...")
        return

    key, value = filters_input.split('=', 1)
    values = [v.strip() for v in value.split(',') if v.strip()]
    filters[key.strip()] = values

    update_values = {}
    for item in update_input.split():
        if '=' not in update_input:
            print(f"badly formatted update value: {item} - use key=value")
            return
        else:
            key, value = update_input.split('=', 1)
            update_values[key.strip()] = value.strip()

    filter_field, filter_values = next(iter(filters.items()))
    update_field, update_value = next(iter(update_values.items()))
    for value in filter_values:
        specieslink.update_records(filters={filter_field: value}, update_values={update_field: update_value}, db_config=db_config, table=table)

In [ ]:
field_input = input("specify the name you want the new field to have (do not create the field manually when running via the pipeline!)").strip()
table = "biodiversity_records"

for default_state, variants in STATE_MAP.items():
    for variant in variants:
        update_input = f"{field_input}={default_state}"
        filters_input = f"stateprovince={variant}"

        print(f"executing specieslink.update_records(filters={filters_input}, update_values={update_input}, table={table})...\n")

        filtering(field_input=field_input,update_input=update_input,filters_input=filters_input,table=table)


In [ ]:
conn = mysql_conn.connect(**db_config)
cursor = conn.cursor()

sql = """
SELECT
    COUNT(*) AS total_records,
    SUM(country_upd IS NOT NULL) AS count_country,
    SUM(country_upd IS NOT NULL AND stateprovince_upd IS NOT NULL) AS count_country_n_state
FROM biodiversity_records
"""

cursor.execute(sql)
total_records, records_country, records_country_state = cursor.fetchone()

cursor.close()
conn.close()

In [ ]:
plt.figure(figsize=(6, 4))

x = [0]

plt.bar(
    x,
    [total_records],
    alpha=0.5,
    width=0.15,
    label='total'
)

plt.bar(
    x,
    [records_country],
    alpha=0.5,
    width=0.15,
    label='with country_upd'
)

plt.bar(
    x,
    [records_country_state],
    width=0.15,
    label='with country_upd and stateprovince_upd'
)

plt.xlim(-0.2, 0.3)
plt.ylabel('total records')
plt.title('sample of records to be used')
plt.legend()

plt.text(0, total_records, str(total_records), ha='center', va='bottom')
plt.text(0, records_country, str(records_country), ha='center', va='bottom')
plt.text(0, records_country_state, str(records_country_state), ha='center', va='bottom')

plt.tight_layout()
plt.show()